# DONT USE THIS ONE
This was my initial attempt but I realized having different .py files for data cleaning makes a lot more sense.

In [ ]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Fixing Daylight Savings.
Making weather data have daylight savings as I don't want to drop rows of the taxi data, but weather data is easier to reason to mix around

In [ ]:
# Weather: Open-Meteo returned fixed UTC-5 (EST) all year despite timezone=America/New_York,
# so convert to real NYC clock time to match the taxi pickup timestamps.
DATA = Path("../data")
wx = pd.read_csv(DATA / "open-meteo-40.74N74.04W51m.csv", skiprows=3)
wx.columns = [c.split(" (")[0] for c in wx.columns]  # drop units, e.g. "temperature_2m (°C)"

wx["time"] = (pd.to_datetime(wx["time"])
                .dt.tz_localize("Etc/GMT+5")  # fixed UTC-5 (Etc sign is inverted)
                .dt.tz_convert("America/New_York")
                .dt.tz_localize(None))

# Fall-back hour (2025-11-02 01:00) now appears twice; trips merge it into one label,
# so do the same: sum amounts over the two hours, average instantaneous values.
wx = wx.groupby("time", as_index=False).agg({
    "temperature_2m": "mean", "apparent_temperature": "mean", "snow_depth": "mean", "is_day": "max",
    "precipitation": "sum", "rain": "sum", "snowfall": "sum", "sunshine_duration": "sum",
})

assert pd.Timestamp("2025-03-09 02:00") not in set(wx["time"])  # spring-forward hour gone
assert wx["time"].is_unique and len(wx) == 8759
wx[wx["time"].between("2025-11-02 00:00", "2025-11-02 03:00")]